In [ ]:
import os
import gc
import json
import math
import time
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Isolated workspace for Mamba
WORK_DIR = Path("/kaggle/working/phase1_mamba")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
SUBMISSION_DIR = WORK_DIR / "submission"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CHECKPOINT = CHECKPOINT_DIR / "checkpoint_latest.pt"
LOG_PATH = WORK_DIR / "training_log.jsonl"

print("=" * 80)
print("CS781 PHASE 1 — MAMBA / SELECTIVE SSM")
print("=" * 80)
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("Work Dir:", WORK_DIR)

In [ ]:
CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]

DATA_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "manifests/train_manifest.jsonl").exists() and (candidate / "tokenizer/legal_bpe_32k.json").exists():
        DATA_ROOT = candidate
        break

assert DATA_ROOT is not None, "Phase 1 training artifacts not found."

TRAIN_MANIFEST = DATA_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = DATA_ROOT / "manifests/validation_manifest.jsonl"
TOKENIZER_PATH = DATA_ROOT / "tokenizer/legal_bpe_32k.json"

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery/indexes/source_index.jsonl"),
    DATA_ROOT / "indexes/source_index.jsonl",
]

INDEX_PATH = None
for candidate in INDEX_CANDIDATES:
    if candidate.exists():
        INDEX_PATH = candidate
        break

assert INDEX_PATH is not None, "Source index file not found."

from tokenizers import Tokenizer, decoders

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()

VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocab size:", VOCAB_SIZE)
print("PAD ID    :", PAD_ID)
print("BOS ID    :", BOS_ID)
print("EOS ID    :", EOS_ID)

In [ ]:
class WorkerSafeDocumentAwarePacker(IterableDataset):
    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=None,
        split="train",
    ):
        super().__init__()
        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or DATA_ROOT)
        self.split = split

        self.index = {}
        with open(self.index_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    self.index[rec["text_hash"]] = {
                        "source_file": rec["source_file"],
                        "byte_offset": rec["byte_offset"],
                        "byte_length": rec["byte_length"],
                    }

        self.manifest = []
        with open(self.manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.manifest.append(json.loads(line))

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

    def _read_source_record(self, rec):
        text_hash = rec["text_hash"]
        idx = self.index[text_hash]
        source_name = idx["source_file"]
        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = Path(self.checkpoint) / "dedup" / source_name
        if not source_path.exists():
            matches = list(Path(self.checkpoint).rglob(Path(source_name).name))
            if len(matches) != 1:
                raise FileNotFoundError(f"Cannot resolve source: {source_name}")
            source_path = matches[0]

        with open(source_path, "rb") as f:
            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        raw = raw.rstrip(b"\r\n")
        return json.loads(raw.decode("utf-8"))

    def _extract_text(self, record):
        if isinstance(record, str):
            return record
        for key in ("text", "content", "document", "body"):
            val = record.get(key)
            if isinstance(val, str):
                return val
        raise KeyError("No text field found.")

    def __iter__(self):
        worker_info = get_worker_info()
        worker_id = 0 if worker_info is None else worker_info.id
        num_workers = 1 if worker_info is None else worker_info.num_workers

        for manifest_idx in range(worker_id, len(self.manifest), num_workers):
            rec = self.manifest[manifest_idx]
            source_record = self._read_source_record(rec)
            text = self._extract_text(source_record)
            
            ids = self.tokenizer.encode(text).ids
            token_ids = [self.bos_id] + list(map(int, ids)) + [self.eos_id]

            for start in range(0, len(token_ids), self.context_length):
                chunk = token_ids[start:start + self.context_length]
                if len(chunk) < self.context_length:
                    chunk = chunk + [self.pad_id] * (self.context_length - len(chunk))

                input_ids = torch.tensor(chunk, dtype=torch.long)
                labels = input_ids.clone()
                labels[labels == self.pad_id] = -100
                attention_mask = (input_ids != self.pad_id).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
NUM_WORKERS = 2

train_dataset = WorkerSafeDocumentAwarePacker(TRAIN_MANIFEST, INDEX_PATH, tokenizer, CONTEXT_LENGTH, DATA_ROOT, "train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

print("✓ Streaming Dataloader initialized")

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        output = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return output * self.weight


class MambaBlock(nn.Module):
    def __init__(
        self,
        d_model: int = 768,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        dt_rank: int = None,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = self.expand * self.d_model
        self.dt_rank = math.ceil(self.d_model / 16) if dt_rank is None else dt_rank

        self.norm = RMSNorm(d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )

        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # Initialize A matrix (S4D real initialization)
        A = torch.repeat_interleave(
            torch.arange(1, self.d_state + 1, dtype=torch.float32),
            repeats=self.d_inner,
        ).reshape(self.d_inner, self.d_state)
        self.A_log = nn.Parameter(torch.log(A))

        # D skip parameter
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # Initialize dt_proj bias
        dt_init_std = self.dt_rank**-0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(
            torch.rand(self.d_inner) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)
        )
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)

        # 1. Input projection and gate split
        xz = self.in_proj(x_norm)
        x_proj, z = xz.chunk(2, dim=-1)

        # 2. Causal 1D Convolution
        x_conv = self.conv1d(x_proj.transpose(1, 2))[:, :, :x.size(1)].transpose(1, 2)
        x_ssm = F.silu(x_conv)

        # 3. Input-dependent parameter generation
        x_dbl = self.x_proj(x_ssm)
        dt, B_t, C_t = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))

        # 4. Selective Scan (Causal recurrence)
        A = -torch.exp(self.A_log.float())
        B_sz, T_sz, _ = x_ssm.shape

        h = torch.zeros((B_sz, self.d_inner, self.d_state), device=x.device, dtype=x_ssm.dtype)
        ys = []
        A_dtype = A.to(dtype=x_ssm.dtype)

        for t in range(T_sz):
            dt_t = dt[:, t, :, None]
            B_t_t = B_t[:, t, None, :]
            C_t_t = C_t[:, t, None, :]
            u_t = x_ssm[:, t, :, None]

            dA = torch.exp(dt_t * A_dtype)
            dB = dt_t * B_t_t

            h = dA * h + dB * u_t
            y_t = (h * C_t_t).sum(dim=-1)
            ys.append(y_t)

        y_ssm = torch.stack(ys, dim=1)
        y_ssm = y_ssm + x_ssm * self.D

        # 5. Gating & Output projection
        y_gated = y_ssm * F.silu(z)
        out = self.out_proj(y_gated)

        return out + residual


class MambaLegalLM(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        d_model=768,
        n_layer=16,
        d_state=16,
        d_conv=4,
        expand=2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            MambaBlock(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, labels=None, **kwargs):

        x = self.embedding(input_ids)

        # Gradient checkpoint each Mamba block during training.
        # This does not change the MambaBlock itself.
        for layer in self.layers:
            if self.training and torch.is_grad_enabled():
                x = torch.utils.checkpoint.checkpoint(
                    layer,
                    x,
                    use_reentrant=False,
                )
            else:
                x = layer(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None

        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        class Output:
            def __init__(self, logits, loss):
                self.logits = logits
                self.loss = loss

        return Output(logits=logits, loss=loss)

print("✓ Mamba architecture classes defined")

In [ ]:
model = MambaLegalLM(
    vocab_size=VOCAB_SIZE,
    d_model=768,
    n_layer=16,
    d_state=16,
    d_conv=4,
    expand=2,
).to(DEVICE)

param_count = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {param_count:,}")
assert 50_000_000 <= param_count <= 300_000_000, f"Parameter count outside 50M–300M range: {param_count:,}"
print("✓ Parameter count is within 50M–300M requirement")

# Sanity Test
model.eval()
test_seq_len = 128
test_input = torch.randint(0, VOCAB_SIZE, (1, test_seq_len), device=DEVICE, dtype=torch.long)
test_labels = test_input.clone()

with torch.no_grad():
    out = model(test_input, labels=test_labels)

print("Sanity Test Input Shape :", tuple(test_input.shape))
print("Sanity Test Output Shape:", tuple(out.logits.shape))
print("Sanity Test Loss        :", out.loss.item())

assert out.logits.shape == (1, test_seq_len, VOCAB_SIZE)
assert torch.isfinite(out.loss)
print("✓ Mamba Forward Pass Sanity Test PASSED")

In [ ]:
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
GRAD_ACCUM = 8
TOTAL_STEPS = 1000
CHECKPOINT_EVERY = 50
LOG_EVERY = 10

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Optimizer : AdamW")
print("LR        :", LEARNING_RATE)
print("Weight Dec:", WEIGHT_DECAY)
print("AMP       :", USE_AMP)

In [ ]:
# ============================================================
# MAMBA — 4096 TOKEN TRAINING MEMORY TEST
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model.train()
optimizer.zero_grad(set_to_none=True)

test_seq_len = 4096

input_ids = torch.randint(
    0,
    VOCAB_SIZE,
    (1, test_seq_len),
    device=DEVICE,
    dtype=torch.long,
)

labels = input_ids.clone()

with torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
    enabled=USE_AMP,
):
    out = model(
        input_ids,
        labels=labels,
    )
    loss = out.loss / GRAD_ACCUM

scaler.scale(loss).backward()

scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    MAX_GRAD_NORM,
)

scaler.step(optimizer)
scaler.update()

optimizer.zero_grad(set_to_none=True)

print("Loss:", float(out.loss))
print("Grad norm:", float(grad_norm))
print(
    "Allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

assert torch.isfinite(out.loss)
assert torch.isfinite(grad_norm)

print("✓ 4096-token training-memory test passed")

del out, loss, input_ids, labels

gc.collect()
torch.cuda.empty_cache()

In [ ]:
print("d_model:", model.layers[0].d_model)
print("n_layer:", len(model.layers))
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def save_checkpoint(step, reason="periodic"):
    checkpoint = {
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "reason": reason,
        "config": {
            "vocab_size": VOCAB_SIZE,
            "d_model": 768,
            "n_layer": 16,
        },
    }
    temp_path = CHECKPOINT_DIR / "checkpoint_tmp.pt"
    torch.save(checkpoint, temp_path)
    temp_path.replace(LATEST_CHECKPOINT)
    size_gb = LATEST_CHECKPOINT.stat().st_size / (1024**3)
    print(f"✓ Checkpoint saved | step={step} | size={size_gb:.3f} GB | reason={reason}")

# Test 1 optimizer step
model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    out = model(input_ids, labels=labels)
    loss = out.loss / GRAD_ACCUM

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("Single Step Loss     :", float(out.loss.detach()))
print("Single Step Grad Norm:", float(grad_norm))
assert torch.isfinite(out.loss)
assert torch.isfinite(grad_norm)

save_checkpoint(1, reason="sanity_check")
print("✓ ONE REAL OPTIMIZATION STEP & CHECKPOINT TEST PASSED")

In [ ]:
ckpt = torch.load(LATEST_CHECKPOINT, map_location="cpu")
print("Checkpoint step:", ckpt["step"])

In [ ]:
START_STEP = 1

if LATEST_CHECKPOINT.exists():
    ckpt = torch.load(LATEST_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    START_STEP = ckpt["step"]
    print(f"Resuming training from step {START_STEP}...")

model.train()
train_iterator = iter(train_loader)
running_loss = 0.0
start_time = time.time()

for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(input_ids, labels=labels)
            loss = out.loss
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        accumulated_loss += loss.detach().float().item()

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()

    mean_loss = accumulated_loss / GRAD_ACCUM
    running_loss += mean_loss

    if step % LOG_EVERY == 0:
        elapsed = time.time() - start_time
        avg_loss = running_loss / LOG_EVERY
        ppl = math.exp(min(avg_loss, 20))
        
        log_rec = {
            "step": step,
            "loss": avg_loss,
            "perplexity": ppl,
            "grad_norm": float(grad_norm),
            "elapsed_seconds": elapsed,
        }
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_rec) + "\n")

        print(f"step={step:4d} | loss={avg_loss:.4f} | ppl={ppl:.2f} | grad={float(grad_norm):.3f} | time={elapsed/60:.1f}m")
        running_loss = 0.0

    if step % CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, reason="final" if step == TOTAL_STEPS else "periodic")

print("=" * 80)
print("MAMBA TRAINING COMPLETE — Step 1000 Reached")
print("=" * 80)

In [ ]:
import time
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

torch.cuda.synchronize()
t0 = time.time()

with torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
    enabled=USE_AMP,
):
    out = model(input_ids, labels=labels)
    loss = out.loss / GRAD_ACCUM

scaler.scale(loss).backward()

scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    MAX_GRAD_NORM,
)

scaler.step(optimizer)
scaler.update()

torch.cuda.synchronize()
elapsed = time.time() - t0

optimizer.zero_grad(set_to_none=True)

print(f"One micro-step time: {elapsed:.1f} sec")
print(f"Loss: {float(out.loss):.4f}")
print(f"Grad norm: {float(grad_norm):.4f}")
print(f"Estimated time for 8 micro-steps: {elapsed*8/60:.1f} min")

In [ ]:
# ============================================================
# VERIFY MAMBA MODEL
# ============================================================

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Parameters:", sum(p.numel() for p in model.parameters()))
print("d_model:", model.layers[0].d_model)
print("n_layer:", len(model.layers))

assert model.layers[0].d_model == 768
assert len(model.layers) == 16

print("✓ Model configuration verified")